# Notebook 15 – Complete Feature Engineering Workflow

## Objective

To perform a complete feature engineering workflow using the cleaned Telco Customer Churn dataset from Sprint 5.

## Workflow

1. Load Clean Dataset
2. Understand Existing Features
3. Identify Potential Feature Gaps
4. Create Numerical Features
5. Create Categorical Features
6. Create Date/Time Features
7. Create Aggregation Features
8. Create Interaction Features
9. Perform Feature Selection
10. Analyze Feature Importance
11. Check for Feature Leakage
12. Remove Unnecessary Features
13. Create Final Feature Dataset
14. Document All Feature Engineering Decisions

## Final Output

Clean Dataset → Engineered Dataset → Selected Features → Final ML-Ready Feature Set



## Step 1 – Understand

### 1.Load Clean Dataset

A clean dataset is a dataset that has already undergone basic preprocessing.

It should contain:
- Correct data types
- Handled missing values
- Removed duplicate records
- Consistent column values
- A clearly identified target column

In this project, we use the cleaned Telco Customer Churn dataset from Sprint 5.

The target column is Churn.



## Step 2 – Demonstrate

### Example

Consider a customer dataset containing:

- Customer ID
- Gender
- Tenure
- Monthly Charges
- Total Charges
- Contract
- Churn

Before feature engineering, we must understand the dataset structure and verify that the data is ready.

Business Example:

A telecom company wants to predict whether a customer will leave the service.

The cleaned dataset provides the existing customer information required for this task.



## Step 3 – Implement

### Import Libraries and Load Clean Dataset

We use Pandas to load and inspect the cleaned dataset.

We use NumPy for numerical operations.

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from tkinter import Tk, filedialog

# Select the cleaned Sprint 5 CSV dataset
root = Tk()
root.withdraw()

file_path = filedialog.askopenfilename(
    title="Select Sprint 5 Cleaned Dataset",
    filetypes=[("CSV Files", "*.csv")]
)

root.destroy()

if not file_path:
    raise FileNotFoundError("Please select the cleaned CSV file.")

df = pd.read_csv(file_path)

print("Dataset Loaded Successfully!")
print("Dataset Name:", Path(file_path).name)
print("Dataset Shape:", df.shape)

display(df.head())

### Output Explanation

- Dataset Loaded Successfully: Confirms that the CSV file was loaded.
- Dataset Shape: Displays the number of rows and columns.
- Head: Displays the first five records.

The loaded dataset is stored in the variable df.

## Step 1 – Understand

### 2.Understand Existing Features

Existing features are the original columns available in the dataset.

They provide information about customers, services, contracts and billing.

Examples:

- tenure: Customer relationship duration in months.
- MonthlyCharges: Monthly customer bill.
- TotalCharges: Total accumulated charges.
- Contract: Customer contract type.
- InternetService: Internet service category.
- PaymentMethod: Customer payment method.

Understanding existing features helps identify opportunities for creating new features.



## Step 2 – Demonstrate

### Example

Consider a customer with:

- Tenure = 24 months
- MonthlyCharges = 100
- TotalCharges = 2400
- Contract = One year

Business Understanding:

The customer has stayed for two years and pays 100 per month.

Using these existing features, we can create additional information such as tenure in years and customer lifecycle groups.




## Step 3 – Implement

### Explore Existing Features

We inspect the column names, data types, numerical values and categorical values.

In [ ]:
print("Dataset Columns:")
print(df.columns.tolist())

print("\nDataset Information:")
df.info()

print("\nNumerical Feature Summary:")
display(df.describe())

print("\nCategorical Feature Summary:")
display(df.describe(include="object"))

print("\nFirst Five Records:")
display(df.head())

### Output Explanation

- Columns: Shows all available features.
- Info: Displays data types and non-null values.
- Describe: Provides numerical statistics.
- Categorical Summary: Displays unique values and category frequencies.

This helps us understand the existing dataset before creating new features.

# Topic 3 – Identify Potential Feature Gaps


## Step 1 – Understand

Feature gaps are missing pieces of information that may help a machine learning model understand customer behaviour better.

We identify:
- Features that are missing.
- Features that can be derived from existing columns.
- Features that may improve prediction.
- Features that are not useful for prediction.

Example:
The dataset contains Tenure and MonthlyCharges, but it does not directly contain TenureYears or AverageMonthlyCost.

We can create these features using existing columns.



## Step 2 – Demonstrate

Consider a telecom customer:

- Tenure = 24 months
- MonthlyCharges = 100
- TotalCharges = 2400

Potential feature gaps:

- TenureYears = 24 / 12 = 2 years
- AverageChargesPerMonth = 2400 / 24 = 100
- TenureMonthlyCharge = 24 × 100 = 2400

These additional features may help us understand customer usage and spending patterns.


## Step 3 – Implement

Check the existing columns and identify possible feature gaps.

We will check which source columns are available before creating new features.

In [ ]:
available_columns = df.columns.tolist()

potential_features = {
    "TenureYears": ["tenure"],
    "AverageChargesPerMonth": ["TotalCharges", "tenure"],
    "TenureMonthlyCharge": ["tenure", "MonthlyCharges"],
    "HasFamily": ["Partner", "Dependents"],
    "IsMonthToMonth": ["Contract"]
}

print("Potential Feature Gaps:\n")

for feature, required_columns in potential_features.items():
    missing_columns = [
        column for column in required_columns
        if column not in available_columns
    ]

    if not missing_columns:
        print(f"{feature}: Can be created")
    else:
        print(f"{feature}: Cannot be created - missing {missing_columns}")

### Output Explanation

The output identifies which potential features can be created from the existing dataset.

If all required source columns are available, the feature can be created.

If a required column is missing, we should not create that feature using invented values.

This step helps us plan feature engineering safely.

# Topic 4 – Create Numerical Features



## Step 1 – Understand

Numerical features are new numeric columns created using existing numerical data.

They help the model understand customer spending, usage, duration, and numerical relationships.

Examples:
- TenureYears
- AverageChargesPerMonth
- TenureMonthlyCharge



## Step 2 – Demonstrate

Consider a customer:

- Tenure = 24 months
- MonthlyCharges = 100
- TotalCharges = 2400

Calculations:

TenureYears = 24 / 12 = 2

AverageChargesPerMonth = 2400 / 24 = 100

TenureMonthlyCharge = 24 × 100 = 2400

These features provide additional numerical information about the customer.



## Step 3 – Implement

Create numerical features only when their required columns are available.

For customers with zero tenure, avoid division by zero when calculating average charges.

In [ ]:
engineered_df = df.copy()

if "tenure" in engineered_df.columns:
    engineered_df["TenureYears"] = engineered_df["tenure"] / 12

if {"TotalCharges", "tenure"}.issubset(engineered_df.columns):
    total_charges = pd.to_numeric(
        engineered_df["TotalCharges"], errors="coerce"
    )
    tenure = pd.to_numeric(
        engineered_df["tenure"], errors="coerce"
    )

    engineered_df["AverageChargesPerMonth"] = (
        total_charges / tenure.replace(0, np.nan)
    )

if {"tenure", "MonthlyCharges"}.issubset(engineered_df.columns):
    engineered_df["TenureMonthlyCharge"] = (
        pd.to_numeric(engineered_df["tenure"], errors="coerce")
        * pd.to_numeric(engineered_df["MonthlyCharges"], errors="coerce")
    )

print("Numerical Features Created:")
display(
    engineered_df[
        [
            column for column in [
                "TenureYears",
                "AverageChargesPerMonth",
                "TenureMonthlyCharge"
            ]
            if column in engineered_df.columns
        ]
    ].head()
)

### Output Explanation

The output displays the newly created numerical features.

- TenureYears represents the customer's tenure in years.
- AverageChargesPerMonth represents average charges per month.
- TenureMonthlyCharge represents tenure multiplied by monthly charges.

These features may provide useful information for customer churn prediction.

# Topic 5 – Create Categorical Features


## Step 1 – Understand

Categorical feature engineering creates new categories from existing columns.

It helps convert numerical values and existing categories into meaningful business groups.

Examples:
- TenureGroup
- IsMonthToMonth
- HasFamily


## Step 2 – Demonstrate

Example:

A customer has 24 months of tenure.

We can classify the customer into:

- New Customer: 0–12 months
- Medium-Term Customer: More than 12–48 months
- Long-Term Customer: More than 48 months

We can also create IsMonthToMonth from the Contract column.



## Step 3 – Implement

Create categorical features using existing tenure, contract, partner, and dependent columns.

Use conditions to create meaningful customer groups.

In [ ]:
if "tenure" in engineered_df.columns:
    engineered_df["TenureGroup"] = pd.cut(
        pd.to_numeric(engineered_df["tenure"], errors="coerce"),
        bins=[-1, 12, 48, float("inf")],
        labels=["New", "Medium-Term", "Long-Term"]
    )

if "Contract" in engineered_df.columns:
    engineered_df["IsMonthToMonth"] = (
        engineered_df["Contract"] == "Month-to-month"
    ).astype(int)

if {"Partner", "Dependents"}.issubset(engineered_df.columns):
    engineered_df["HasFamily"] = (
        (engineered_df["Partner"] == "Yes")
        | (engineered_df["Dependents"] == "Yes")
    ).astype(int)

print("Categorical Features Created:")
display(
    engineered_df[
        [
            column for column in [
                "TenureGroup",
                "IsMonthToMonth",
                "HasFamily"
            ]
            if column in engineered_df.columns
        ]
    ].head()
)

### Output Explanation

The output displays the newly created categorical features.

- TenureGroup groups customers based on their tenure.
- IsMonthToMonth identifies customers with month-to-month contracts.
- HasFamily identifies customers who have a partner or dependents.

These features help describe customer behaviour in a business-friendly way.

# Topic 6 – Create Date/Time Features


## Step 1 – Understand

Date and time feature engineering extracts useful information from date and time columns.

Common date/time features include:
- Year
- Month
- Day
- Day of Week
- Quarter
- Customer Duration

These features can help identify seasonal and time-based patterns.



## Step 2 – Demonstrate

Consider a customer registration date:

2024-05-15

We can extract:

- Year = 2024
- Month = 5
- Day = 15
- Day of Week = Wednesday

The Telco Customer Churn dataset may not contain a calendar registration date.

If no date column exists, we should document that limitation instead of creating fake dates.



## Step 3 – Implement

First, identify date-like columns.

If a suitable date column exists, convert it into datetime format and extract date components.

If no date column exists, document that no calendar date features can be created.

In [ ]:
date_columns = [
    column for column in engineered_df.columns
    if "date" in column.lower()
    or "datetime" in column.lower()
]

if date_columns:
    for column in date_columns:
        engineered_df[column] = pd.to_datetime(
            engineered_df[column],
            errors="coerce"
        )

        engineered_df[f"{column}_Year"] = engineered_df[column].dt.year
        engineered_df[f"{column}_Month"] = engineered_df[column].dt.month
        engineered_df[f"{column}_Day"] = engineered_df[column].dt.day
        engineered_df[f"{column}_DayOfWeek"] = (
            engineered_df[column].dt.dayofweek
        )

    print("Date/Time Features Created:")
    display(engineered_df[date_columns].head())

else:
    print("No calendar date/time columns found.")
    print("No artificial date features were created.")

if "tenure" in engineered_df.columns:
    engineered_df["CustomerDurationMonths"] = engineered_df["tenure"]

    print("\nCustomer Duration Feature:")
    display(engineered_df[["CustomerDurationMonths"]].head())

### Output Explanation

If date columns are available, the output displays their extracted date components.

If no date columns are available, the notebook reports that no calendar date features were created.

CustomerDurationMonths is derived from tenure. It represents customer duration, not a calendar date.

# Topic 7 – Create Aggregation Features


## Step 1 – Understand

Aggregation features summarize data using calculations such as:

- Count
- Sum
- Mean
- Median
- Minimum
- Maximum

In customer churn prediction, we can calculate the average monthly charges for each contract type.

Important: Aggregation values used for machine learning must be learned from training data only. This prevents information from the test dataset from influencing the model.



## Step 2 – Demonstrate

Example:

Suppose customers have the following contract types and monthly charges:

| Contract | MonthlyCharges |
|---|---:|
| Month-to-month | 80 |
| Month-to-month | 100 |
| One year | 60 |
| One year | 80 |

Average monthly charges:

- Month-to-month = 90
- One year = 70

These group averages provide additional information about customer spending patterns.



## Step 3 – Implement

First, separate the target column from the feature data.

Then split the dataset into training and testing sets.

Calculate contract-based and internet-service-based average charges using only the training data.

Apply the training averages to both datasets.

In [ ]:
from sklearn.model_selection import train_test_split

target_column = "Churn"

if target_column not in engineered_df.columns:
    raise KeyError(
        "The target column 'Churn' was not found. "
        "Check your cleaned Sprint 5 dataset."
    )

# Separate target and features
y = engineered_df[target_column].copy()
X = engineered_df.drop(columns=[target_column]).copy()

# Remove customer ID from model features
if "customerID" in X.columns:
    X = X.drop(columns=["customerID"])

# Split before learning aggregation values
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Calculate aggregation values using training data only
aggregation_columns = []

if {"Contract", "MonthlyCharges"}.issubset(X_train.columns):
    contract_means = X_train.groupby("Contract")["MonthlyCharges"].mean()
    global_mean = X_train["MonthlyCharges"].mean()

    X_train["ContractAverageCharges"] = (
        X_train["Contract"].map(contract_means).fillna(global_mean)
    )
    X_test["ContractAverageCharges"] = (
        X_test["Contract"].map(contract_means).fillna(global_mean)
    )

    aggregation_columns.append("ContractAverageCharges")

if {"InternetService", "MonthlyCharges"}.issubset(X_train.columns):
    internet_means = (
        X_train.groupby("InternetService")["MonthlyCharges"].mean()
    )
    global_mean = X_train["MonthlyCharges"].mean()

    X_train["InternetAverageCharges"] = (
        X_train["InternetService"].map(internet_means).fillna(global_mean)
    )
    X_test["InternetAverageCharges"] = (
        X_test["InternetService"].map(internet_means).fillna(global_mean)
    )

    aggregation_columns.append("InternetAverageCharges")

print("Aggregation Features Created:", aggregation_columns)

display(X_train[aggregation_columns].head())

### Output Explanation

The output displays the aggregation features created from training data.

- ContractAverageCharges represents the average monthly charges for a contract type.
- InternetAverageCharges represents the average monthly charges for an internet service type.

The same training averages are applied to the test dataset.

This prevents aggregation statistics from being learned using test data.

# Topic 8 – Create Interaction Features


## Step 1 – Understand

Interaction features combine two or more existing features to represent their relationship.

They help machine learning models understand how one feature may affect another.

Examples:
- Tenure × MonthlyCharges
- MonthlyCharges × Contract
- Tenure × InternetService



## Step 2 – Demonstrate

Example:

Customer A:
- Tenure = 12 months
- MonthlyCharges = 100

Customer B:
- Tenure = 48 months
- MonthlyCharges = 100

TenureMonthlyCharge:

Customer A = 12 × 100 = 1200

Customer B = 48 × 100 = 4800

The interaction feature combines customer duration and monthly spending.



## Step 3 – Implement

Create numerical interaction features using existing numerical columns.

Create a categorical interaction feature by combining Contract and InternetService when both columns are available.

In [ ]:
if {"tenure", "MonthlyCharges"}.issubset(X_train.columns):
    for dataset in [X_train, X_test]:
        dataset["TenureChargeInteraction"] = (
            pd.to_numeric(dataset["tenure"], errors="coerce")
            * pd.to_numeric(dataset["MonthlyCharges"], errors="coerce")
        )

if {"Contract", "InternetService"}.issubset(X_train.columns):
    for dataset in [X_train, X_test]:
        dataset["ContractInternetInteraction"] = (
            dataset["Contract"].astype(str)
            + "_"
            + dataset["InternetService"].astype(str)
        )

interaction_columns = [
    column for column in [
        "TenureChargeInteraction",
        "ContractInternetInteraction"
    ]
    if column in X_train.columns
]

print("Interaction Features Created:")
display(X_train[interaction_columns].head())

### Output Explanation

The output displays the newly created interaction features.

- TenureChargeInteraction combines tenure and monthly charges.
- ContractInternetInteraction combines contract type and internet service.

These features may help the model identify relationships between customer characteristics.

# Topic 9 – Perform Feature Selection


## Step 1 – Understand

Feature selection is the process of choosing the most useful features for machine learning.

It helps:
- Reduce unnecessary features.
- Improve model efficiency.
- Reduce noise.
- Simplify model training.

Feature selection must be performed using training data only.



## Step 2 – Demonstrate

Suppose a dataset contains 30 features.

Not all 30 features may be useful for predicting customer churn.

Feature selection identifies a smaller group of informative features.

For this notebook, we will use:
- Missing value imputation.
- Numerical scaling.
- Categorical encoding.
- Mutual information feature selection.

The preprocessing and selection steps will be fitted only on training data.



## Step 3 – Implement

Separate numerical and categorical columns.

Use a preprocessing pipeline to:
- Fill missing numerical values.
- Scale numerical features.
- Fill missing categorical values.
- Encode categorical features.

Apply SelectKBest with mutual information to select useful features.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif

# Remove columns that should not be used as model features
unnecessary_columns = [
    column for column in ["customerID"]
    if column in X_train.columns
]

X_train_model = X_train.drop(columns=unnecessary_columns).copy()
X_test_model = X_test.drop(columns=unnecessary_columns).copy()

# Identify numerical and categorical columns
numeric_features = X_train_model.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train_model.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# Fit only on training data
X_train_processed = preprocessor.fit_transform(X_train_model)
X_test_processed = preprocessor.transform(X_test_model)

# Select features using training data
number_of_features = X_train_processed.shape[1]
k_features = min(20, number_of_features)

feature_selector = SelectKBest(
    score_func=mutual_info_classif,
    k=k_features
)

X_train_selected = feature_selector.fit_transform(
    X_train_processed, y_train
)

X_test_selected = feature_selector.transform(X_test_processed)

print("Original Encoded Feature Count:", number_of_features)
print("Selected Feature Count:", X_train_selected.shape[1])
print("Training Shape:", X_train_selected.shape)
print("Testing Shape:", X_test_selected.shape)

### Output Explanation

The output displays the number of encoded features before and after feature selection.

- X_train_selected contains selected training features.
- X_test_selected contains the corresponding selected test features.
- The feature selector learns which features to keep using training data only.

The selected features will be used for feature importance analysis and model training.

# Topic 10 – Analyze Feature Importance


## Step 1 – Understand

Feature importance measures how useful each feature is for making predictions.

It helps us understand which features contribute more to a machine learning model.

A feature with higher importance has a stronger contribution to that particular model.

Feature importance does not prove that a feature causes customer churn.



## Step 2 – Demonstrate

In customer churn prediction, features such as:

- Contract type
- Tenure
- Monthly charges
- Internet service

may help a model distinguish between customers who leave and customers who stay.

A Random Forest model can estimate feature importance based on how it uses features during prediction.



## Step 3 – Implement

Get the names of the encoded features.

Identify the features selected by SelectKBest.

Train a Random Forest classifier using the selected training features.

Display the feature importance scores in descending order.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Get names of all encoded features
encoded_feature_names = preprocessor.get_feature_names_out()

# Get the feature selection mask
selected_mask = feature_selector.get_support()

selected_feature_names = encoded_feature_names[selected_mask]

# Train Random Forest using selected features
importance_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

importance_model.fit(X_train_selected, y_train)

# Create feature importance table
feature_importance = pd.DataFrame({
    "Feature": selected_feature_names,
    "Importance": importance_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

print("Top 15 Important Features:")
display(feature_importance.head(15))

# Visualize the top 15 features
top_features = feature_importance.head(15).sort_values(
    by="Importance"
)

plt.figure(figsize=(10, 6))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.title("Top 15 Feature Importances")
plt.tight_layout()
plt.show()

### Output Explanation

The output displays the feature importance scores from the Random Forest model.

- Higher scores indicate greater importance in this model.
- Lower scores indicate smaller contributions.
- The bar chart makes it easier to compare important features.

Feature importance can vary depending on the model, dataset, and training process.

# Topic 11 – Check for Feature Leakage



## Step 1 – Understand

Feature leakage occurs when a machine learning model receives information that would not be available at the time of prediction.

It can make model performance look better during testing than it will be in real-world use.

Common causes:
- Including the target column in input features.
- Using future information.
- Calculating preprocessing statistics using the full dataset.
- Using test data during feature selection.



## Step 2 – Demonstrate

Suppose we want to predict whether a customer will cancel a telecom service.

If we use a CancellationDate column that is recorded only after the customer cancels, the model receives future information.

This is data leakage.

Similarly, the Churn target column must not be included in the input features.

We must fit preprocessing and feature selection using training data only.




## Step 3 – Implement

Check whether:
- The target column is included in model features.
- Customer ID is included in model features.
- Suspicious future-information columns are present.
- The preprocessing and feature selection steps were fitted only on training data.

In [ ]:
print("Feature Leakage Checks\n")

# Check target column
print("1. Target column included in X_train_model:",
      target_column in X_train_model.columns)

# Check customer ID
print("2. Customer ID included in X_train_model:",
      "customerID" in X_train_model.columns)

# Check suspicious future-information columns
suspicious_keywords = [
    "cancellationdate",
    "terminationdate",
    "cancellationreason",
    "terminationreason",
    "churnstatus"
]

suspicious_columns = [
    column for column in X_train_model.columns
    if any(word in column.lower() for word in suspicious_keywords)
]

print("3. Suspicious columns:", suspicious_columns)

# Confirm the training-only workflow
print("4. Preprocessor fitted using training data only: Yes")
print("5. Feature selector fitted using training data only: Yes")

if (
    target_column not in X_train_model.columns
    and "customerID" not in X_train_model.columns
    and not suspicious_columns
):
    print("\nNo obvious feature leakage detected by these checks.")
else:
    print("\nReview the flagged columns before model training.")

### Output Explanation

The output reports whether the target or customer ID appears in the model input features.

It also checks for suspicious columns that may contain future information.

These checks help identify common leakage risks, but they cannot guarantee that every feature is leakage-free.

Every feature should also be reviewed based on when its information becomes available in a real-world prediction scenario.

# Topic 12 – Remove Unnecessary Features


## Step 1 – Understand

Unnecessary features are columns that do not provide useful information for model prediction.

Examples:
- Customer ID
- Constant columns
- Duplicate information
- Columns containing future information
- Columns that are not relevant to the prediction task

Removing unnecessary features can simplify the dataset and reduce noise.



## Step 2 – Demonstrate

Consider a customer dataset containing:

- customerID
- tenure
- MonthlyCharges
- Contract
- Churn

The customerID column identifies a customer but usually does not explain customer behaviour.

Therefore, we can remove customerID from the model input features.

We must keep Churn separately as the target variable.



## Step 3 – Implement

Identify constant columns using the training dataset.

Remove customerID and constant columns from both training and testing datasets.

Do not remove the target variable from y_train or y_test.

In [ ]:
# Start from the original split datasets
X_train_final_source = X_train.copy()
X_test_final_source = X_test.copy()

# Identify constant columns from training data only
constant_columns = [
    column
    for column in X_train_final_source.columns
    if X_train_final_source[column].nunique(dropna=False) <= 1
]

# Identify ID columns
id_columns = [
    column
    for column in X_train_final_source.columns
    if column.lower() in ["customerid", "customer_id", "id"]
]

# Remove unnecessary columns
columns_to_remove = list(set(constant_columns + id_columns))

X_train_final_source = X_train_final_source.drop(
    columns=columns_to_remove,
    errors="ignore"
)

X_test_final_source = X_test_final_source.drop(
    columns=columns_to_remove,
    errors="ignore"
)

print("Removed Columns:", columns_to_remove)
print("Training Dataset Shape:", X_train_final_source.shape)
print("Testing Dataset Shape:", X_test_final_source.shape)

print("\nRemaining Features:")
print(X_train_final_source.columns.tolist())

### Output Explanation

The output displays the columns removed from the training and testing datasets.

- ID columns are removed because they are identifiers.
- Constant columns are removed because they do not provide variation.
- The target variable remains separate in y_train and y_test.

The resulting datasets contain the remaining candidate features.

# Topic 13 – Create Final Feature Dataset



## Step 1 – Understand

The final feature dataset contains the selected and processed features that are ready for machine learning.

It should contain:
- Useful numerical features.
- Encoded categorical features.
- Selected features.
- No target leakage.
- No unnecessary identifier columns.

The target variable must be stored separately from the input features.



## Step 2 – Demonstrate

The complete workflow:

Clean Dataset
       ↓
Engineered Dataset
       ↓
Selected Features
       ↓
Final ML-Ready Feature Set

The final training and testing datasets must contain the same selected columns and feature order.

The target variable Churn is stored separately.



## Step 3 – Implement

Convert the selected feature arrays into DataFrames.

Use the selected feature names as column names.

Create final training and testing datasets.

Export the final datasets as CSV files.

In [ ]:
# Convert selected arrays to dense arrays when necessary
if hasattr(X_train_selected, "toarray"):
    X_train_array = X_train_selected.toarray()
else:
    X_train_array = np.asarray(X_train_selected)

if hasattr(X_test_selected, "toarray"):
    X_test_array = X_test_selected.toarray()
else:
    X_test_array = np.asarray(X_test_selected)

# Create final feature DataFrames
X_train_final = pd.DataFrame(
    X_train_array,
    columns=selected_feature_names,
    index=X_train.index
)

X_test_final = pd.DataFrame(
    X_test_array,
    columns=selected_feature_names,
    index=X_test.index
)

# Keep target values separate
y_train_final = y_train.copy()
y_test_final = y_test.copy()

# Export final datasets
X_train_final.to_csv("Final_Training_Features.csv", index=False)
X_test_final.to_csv("Final_Testing_Features.csv", index=False)

y_train_final.to_frame(name=target_column).to_csv(
    "Final_Training_Target.csv",
    index=False
)

y_test_final.to_frame(name=target_column).to_csv(
    "Final_Testing_Target.csv",
    index=False
)

print("Final ML-Ready Datasets Created Successfully!")

print("\nTraining Features Shape:", X_train_final.shape)
print("Testing Features Shape:", X_test_final.shape)

print("\nTraining Target Shape:", y_train_final.shape)
print("Testing Target Shape:", y_test_final.shape)

print("\nFinal Training Features:")
display(X_train_final.head())

### Output Explanation

The output displays the final training and testing feature datasets.

Created files:

- Final_Training_Features.csv
- Final_Testing_Features.csv
- Final_Training_Target.csv
- Final_Testing_Target.csv

The feature datasets contain selected and transformed features.

The target datasets contain Churn values separately.

These datasets are ready to be used for machine learning model training and evaluation.

# Topic 14 – Document All Feature Engineering Decisions


## Step 1 – Understand

Feature engineering documentation records the changes made to the original dataset.

It explains:
- Which features were created.
- Which source columns were used.
- Why each feature was created.
- What business meaning the feature provides.
- Whether the feature has any leakage risk.
- Which features were removed.

Documentation makes the workflow easier to understand, maintain, and reproduce.



## Step 2 – Demonstrate

Example documentation:

| Feature | Decision | Reason |
|---|---|---|
| TenureYears | Created | Represents customer duration in years |
| TenureGroup | Created | Groups customers by tenure |
| ContractAverageCharges | Created | Represents average charges for each contract |
| customerID | Removed | Identifier, not a useful model input |
| Churn | Retained separately | Target variable |

This documentation explains the decisions made during feature engineering.



## Step 3 – Implement

Create a feature engineering decision log.

Include:
- Feature name
- Source columns
- Engineering method
- Business meaning
- Decision
- Reason

Export the documentation as a CSV file.

In [ ]:
feature_decisions = [
    {
        "Feature": "TenureYears",
        "SourceColumns": "tenure",
        "Method": "tenure / 12",
        "BusinessMeaning": "Customer duration in years",
        "Decision": "Created",
        "Reason": "Provides tenure information in years"
    },
    {
        "Feature": "AverageChargesPerMonth",
        "SourceColumns": "TotalCharges, tenure",
        "Method": "TotalCharges / tenure",
        "BusinessMeaning": "Average monthly customer charges",
        "Decision": "Created",
        "Reason": "Represents average customer spending"
    },
    {
        "Feature": "TenureMonthlyCharge",
        "SourceColumns": "tenure, MonthlyCharges",
        "Method": "tenure * MonthlyCharges",
        "BusinessMeaning": "Combined tenure and monthly charges",
        "Decision": "Created",
        "Reason": "Captures a numerical relationship"
    },
    {
        "Feature": "TenureGroup",
        "SourceColumns": "tenure",
        "Method": "Categorical binning",
        "BusinessMeaning": "Customer tenure category",
        "Decision": "Created",
        "Reason": "Groups customers by duration"
    },
    {
        "Feature": "IsMonthToMonth",
        "SourceColumns": "Contract",
        "Method": "Binary encoding",
        "BusinessMeaning": "Identifies month-to-month contracts",
        "Decision": "Created",
        "Reason": "Represents contract type"
    },
    {
        "Feature": "HasFamily",
        "SourceColumns": "Partner, Dependents",
        "Method": "Logical OR",
        "BusinessMeaning": "Identifies customers with family connections",
        "Decision": "Created",
        "Reason": "Adds family-related customer information"
    },
    {
        "Feature": "ContractAverageCharges",
        "SourceColumns": "Contract, MonthlyCharges",
        "Method": "Training-data group mean",
        "BusinessMeaning": "Average charges for each contract type",
        "Decision": "Created",
        "Reason": "Captures contract-level spending patterns"
    },
    {
        "Feature": "InternetAverageCharges",
        "SourceColumns": "InternetService, MonthlyCharges",
        "Method": "Training-data group mean",
        "BusinessMeaning": "Average charges for each internet service",
        "Decision": "Created",
        "Reason": "Captures internet-service spending patterns"
    },
    {
        "Feature": "TenureChargeInteraction",
        "SourceColumns": "tenure, MonthlyCharges",
        "Method": "Multiplication",
        "BusinessMeaning": "Interaction between tenure and monthly charges",
        "Decision": "Created",
        "Reason": "Captures relationships between numerical features"
    },
    {
        "Feature": "ContractInternetInteraction",
        "SourceColumns": "Contract, InternetService",
        "Method": "Category combination",
        "BusinessMeaning": "Combined contract and internet service",
        "Decision": "Created",
        "Reason": "Captures combined service information"
    },
    {
        "Feature": "customerID",
        "SourceColumns": "customerID",
        "Method": "Column removal",
        "BusinessMeaning": "Customer identifier",
        "Decision": "Removed from model inputs",
        "Reason": "Identifier is not used as a prediction feature"
    },
    {
        "Feature": "Churn",
        "SourceColumns": "Churn",
        "Method": "Target separation",
        "BusinessMeaning": "Customer churn outcome",
        "Decision": "Retained separately",
        "Reason": "Used as the machine learning target"
    }
]

feature_documentation = pd.DataFrame(feature_decisions)

# Add information about calendar date features
feature_documentation.loc[len(feature_documentation)] = {
    "Feature": "Calendar Date Features",
    "SourceColumns": "Date/Time columns",
    "Method": "Year, Month, Day extraction",
    "BusinessMeaning": "Calendar-based customer information",
    "Decision": "Not created if unavailable",
    "Reason": "No suitable calendar date column was found in the source dataset"
}

feature_documentation.to_csv(
    "Feature_Engineering_Documentation.csv",
    index=False
)

print("Feature Engineering Documentation:")
display(feature_documentation)

print("\nDocumentation saved successfully!")

### Output Explanation

The output displays the feature engineering decision log.

It documents the features created, their source columns, engineering methods, business meanings, and decisions.

The documentation is saved as:

Feature_Engineering_Documentation.csv

This file provides a record of the complete feature engineering process.

# Final Workflow Summary

This notebook completed the following feature engineering workflow:

1. Loaded the clean Sprint 5 dataset.
2. Understood the existing features.
3. Identified potential feature gaps.
4. Created numerical features.
5. Created categorical features.
6. Checked for date/time features.
7. Created aggregation features.
8. Created interaction features.
9. Performed feature selection.
10. Analyzed feature importance.
11. Checked for feature leakage.
12. Identified and removed unnecessary features.
13. Created final ML-ready feature datasets.
14. Documented feature engineering decisions.

## Final Pipeline

Clean Dataset
        ↓
Engineered Dataset
        ↓
Selected Features
        ↓
Final ML-Ready Feature Set

## Final Deliverables

- Final_Training_Features.csv
- Final_Testing_Features.csv
- Final_Training_Target.csv
- Final_Testing_Target.csv
- Feature_Engineering_Documentation.csv

## Conclusion

The complete feature engineering workflow transforms the cleaned dataset into selected and processed features suitable for machine learning.

The final training and testing datasets are kept separate to support reliable model development and evaluation.